In [3]:
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
import os
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.tools import tool,tool_node

load_dotenv()

True

In [4]:
## State Definition
class BlogState(TypedDict):
     query: str
     search_results: str
     analysis: str
     final_report: str


In [5]:
# LLM initialization

llm_model = ChatGroq(
    model="openai/gpt-oss-20b"
)

In [6]:
# create tools
# web search tools
web_search_tool = TavilySearch(
    max_results=3
)

In [7]:
# First Node
def search_info_agent(state:BlogState):
    query = state["query"]

    search_results = web_search_tool.invoke(query)

    return {
        "search_results": str(search_results),
    }


In [10]:
def analysis_agent(state:BlogState):

    search_results = state["search_results"]

    ai_response = llm_model.invoke(
        f"""
        Analyze these results:
        {search_results}

        Extract:
        - key insights
        - trends
        - important facts
        - risks

        """
    )
    return {
        "analysis": str(ai_response.content),
    }

In [11]:
def report_node(state: BlogState):

    analysis = state["analysis"]

    response = llm_model.invoke(
        f"""
        Create a professional research report using:

        {analysis}

        Include:
        - Introduction
        - Main Findings
        - Future Scope
        - Conclusion
        """
    )

    return {
        "final_report": response.content
    }

In [12]:
builder = StateGraph(BlogState)

In [13]:
# Adding Nodes

builder.add_node("search", search_info_agent)

builder.add_node("analysis", analysis_agent)

builder.add_node("report", report_node)

In [14]:
builder.add_edge(START, "search")

builder.add_edge("search", "analysis")

builder.add_edge("analysis", "report")

builder.add_edge("report", END)

In [16]:
graph = builder.compile()

In [17]:
result = graph.invoke({
    "query": "Future of autonomous AI agents"
})

In [18]:
result

{'query': 'Future of autonomous AI agents',
 'search_results': '{\'query\': \'Future of autonomous AI agents\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'url\': \'https://www.c-sharpcorner.com/article/the-future-of-autonomous-ai-agents-in-software-engineering/\', \'title\': \'The Future of Autonomous AI Agents in Software Engineering\', \'content\': \'# The Future of Autonomous AI Agents in Software Engineering. Modern AI systems are evolving into autonomous agents capable of planning tasks, writing code, debugging applications, analyzing logs, interacting with APIs, managing workflows, and even collaborating with other AI agents. The software industry is now entering the era of Autonomous AI Agents. From AI-powered DevOps pipelines to autonomous testing systems and intelligent coding assistants, AI agents are becoming deeply integrated into modern engineering workflows. In this article, we will explore how autonomous AI agents work, why they are